In [0]:
dbutils.widgets.text(name="env", defaultValue="", label="Enter the envoronment in lower case")
db = dbutils.widgets.get("env")

In [0]:
%run "./commons"

# Reading from bronze raw_road table


In [0]:
def read_BronzeRawRoadTable(environment):
    print("Reading bronze table", end=' ')
    df_BronzeRoad = (spark.readStream.table(f"{environment}_catalog.bronze.`raw-roads`"))
    print(f'Reading {environment}_catalog.bronze.`raw-roads` successful')
    return df_BronzeRoad

##Creating road category name

In [0]:
def road_category(df):
    print("Creating road category name", end=' ')
    d_road_category = df.withColumn('Road_Category Name', when(df.Road_Category == 'TA', 'Class A Trunk Road')
                                    .when(df.Road_Category == 'TM', 'Class A Trunk Motorway')
                                    .when(df.Road_Category == 'PA', 'Class A Principle Road')
                                    .when(df.Road_Category == 'PM', 'Class A Principle Motorway')
                                    .when(df.Road_Category == 'M', 'Clas B Road')
                                    .otherwise('Unknown'))
    print("Successful")
    return d_road_category

## Creating Road Type Name

In [0]:
from pyspark.sql.functions import col, when

def road_type(df):
    print("Creating road type name", end=' ')
    df_road_type = df.withColumn('Road_Type', 
                                 when(col('Road_Category').like('%Class A%'), 'Major Road')
                                .when(col('Road_Category').like('%Class B%'), 'Minor Road')
                                .otherwise('Unknown'))
    print("Successful")
    return df_road_type

##Writing data to Silver road data Table

In [0]:

def Write_Roads_Silver_Table(StreamingDF, environment):
    print("Writing Silver Table", end=' ')
    
    # Clear checkpoint if it exists to avoid conflicts
    checkpoint_path = checkpoint + "/SilverTrafficcloud/Checkpt/"
    dbutils.fs.rm(checkpoint_path, True)
    
    write_StreamSilver = (
        StreamingDF.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .queryName("SilverTrafficWriteStreamf")
        .trigger(availableNow=True)
        .toTable(f"{environment}_catalog.silver.Silver_Roads")
    )
    
    write_StreamSilver.awaitTermination()
    print(f'Writing {environment}_catalog.silver.Silver_Roads successful')

In [0]:
# Reading bronze raw_road table
df_roads = read_BronzeRawRoadTable(env)

# drop duplicate rows as defined in commons
df_dedup = remove_Dups(df_roads)  

# replace or drop nulls as defined in commons
Allcolumns = df_dedup.schema.names
df_clean = handle_NULLs(df_dedup, Allcolumns) 

# Creating road category name
df_road_category = road_category(df_clean)

# Creating road type name
df_road_type = road_type(df_road_category)             

# Writing data to Silver road data Table
Write_Roads_Silver_Table(df_dedup, db)

In [0]:
%sql
SELECT COUNT(*) FROM `dev_catalog`.`silver`.`Silver_roads` LIMIT 10